In [150]:
# ==========================================
# 1. CONFIGURATION & SETUP
# ==========================================
import os
import copy
import numpy as np
import pandas as pd
import open3d as o3d

WORKPIECE = "test_5_090angle"
SIM_DIR = f"viewpoints_candidate/testing_data/{WORKPIECE}"
OUT_DIR = f"simulation/{WORKPIECE}"

if not os.path.exists(OUT_DIR):
    os.makedirs(OUT_DIR)

TARGET_VIEWPOINT = 4

# ------------------------------------------
# NOISE PARAMETERS
# ------------------------------------------
BASE_SIGMA_XY = 1#1.0  # Base error on X/Y axis (Gaussian)
BASE_SIGMA_Z = 2#2.0   # Base error on Depth (Z) axis (Gaussian)
K_NEIGHBORS = 300#300    # How many points to average over (Controls how large the 'waves' are)
SIGMA_AOI = 30#30.0     # Controls how fast dropout increases. Lower = drops out faster at lower angles


In [151]:
# ==========================================
# 2. MANUAL MATHEMATICAL DROPOUT MODEL
# ==========================================
# We model the probability of a point SURVIVING as a Gaussian curve.
# Survival is highest at 0 degrees (Face-on), and drops as AoI increases.
# Formula: Survival(x) = exp(-x^2 / (2 * sigma^2))
# Dropout(x) = 1.0 - Survival(x)

def calculate_dropout_probability(aoi_degrees):
    if SIGMA_AOI <= 0:
        return 0.0  # Disable dropout completely if sigma is 0!
    # Calculates the percentage of points that should drop out (0.0 to 1.0)
    survival = np.exp(-(aoi_degrees**2) / (2 * (SIGMA_AOI**2)))
    return 1.0 - survival

print(f"Manual Mathematical Dropout Model configured (Sigma = {SIGMA_AOI} degrees)")


Manual Mathematical Dropout Model configured (Sigma = 30 degrees)


In [152]:
# ==========================================
# 3. LOAD CAD & COMPUTE AOI
# ==========================================
print(f"Loading Simulated CAD for Viewpoint {TARGET_VIEWPOINT}...")

sim_path = os.path.join(SIM_DIR, f"viewpoint_simulated_{TARGET_VIEWPOINT}.pcd")
pose_path = os.path.join(SIM_DIR, f"viewpoint_pose_{TARGET_VIEWPOINT}.npy")

if not os.path.exists(sim_path):
    print(f"File not found: {sim_path}")
else:
    pcd_ideal = o3d.io.read_point_cloud(sim_path)
    
    # Ensure normals exist
    if not pcd_ideal.has_normals():
        pcd_ideal.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=10.0, max_nn=30))
        pcd_ideal.orient_normals_towards_camera_location(camera_location=np.array([0., 0., 0.]))
        # Flip normals to point OUT of surface
        normals = np.asarray(pcd_ideal.normals)
        pcd_ideal.normals = o3d.utility.Vector3dVector(-normals)
        
    # Transform CAD into Camera Space
    T_cam_to_obj = np.load(pose_path)
    T_obj_to_cam = np.linalg.inv(T_cam_to_obj)
    pcd_ideal.transform(T_obj_to_cam)
    
    print(f"Loaded and transformed {len(pcd_ideal.points)} points into Camera Space.")


Loading Simulated CAD for Viewpoint 4...
Loaded and transformed 15357 points into Camera Space.


In [153]:
# ==========================================
# 4. APPLY MANUAL DROPOUT & WAVY FIRMWARE NOISE
# ==========================================
import math
import random
points = np.asarray(pcd_ideal.points)
normals = np.asarray(pcd_ideal.normals)

print("Calculating AoI...")
camera_dir = -points
camera_dir_norms = np.linalg.norm(camera_dir, axis=1, keepdims=True)
camera_dir_norm = camera_dir / camera_dir_norms

normal_norms = np.linalg.norm(normals, axis=1, keepdims=True)
normal_norm_vecs = normals / normal_norms

cos_thetas = np.clip(np.sum(camera_dir_norm * normal_norm_vecs, axis=1), -1.0, 1.0)
aois = np.degrees(np.arccos(cos_thetas))
aois = np.where(aois > 90, 180 - aois, aois)

# Group points into 1-degree bins to force exact percentages!
points_by_bin = {i: [] for i in range(91)}
# Keep track of original indices so we can apply noise efficiently later
idx_by_bin = {i: [] for i in range(91)}
for i in range(len(points)):
    bin_idx = int(np.round(aois[i]))
    if bin_idx > 90: bin_idx = 90
    points_by_bin[bin_idx].append(points[i])
    idx_by_bin[bin_idx].append(i)

surviving_indices = []
dropped_count = 0

if SIGMA_AOI <= 0:
    print("SIGMA_AOI is 0. Bypassing Dropout!")
    surviving_indices = list(range(len(points)))
else:
    print("Dropping points based on Gaussian Model...")
    for bin_idx, p_idx_list in idx_by_bin.items():
        if len(p_idx_list) == 0:
            continue
            
        prob = calculate_dropout_probability(bin_idx)
        num_to_drop = int(len(p_idx_list) * prob)
        
        np.random.shuffle(p_idx_list)
        surviving = p_idx_list[num_to_drop:]
        surviving_indices.extend(surviving)
        dropped_count += num_to_drop

print(f"Dropout Complete: {dropped_count} points deleted.")

print(f"Generating Raw TV-Static Gaussian Noise...")
raw_noise_x = np.random.normal(0, BASE_SIGMA_XY, size=len(points))
raw_noise_y = np.random.normal(0, BASE_SIGMA_XY, size=len(points))
raw_noise_z = np.random.normal(0, BASE_SIGMA_Z, size=len(points))
raw_noise = np.column_stack((raw_noise_x, raw_noise_y, raw_noise_z))

smoothed_noise = np.zeros_like(raw_noise)

if K_NEIGHBORS <= 0:
    print("K_NEIGHBORS is 0. Bypassing Firmware Blur (Using Raw Static Noise)...")
    smoothed_noise = raw_noise
else:
    print("Applying Spatial Moving Average (Simulating Firmware Blur for Wavy Effect)...")
    kdtree = o3d.geometry.KDTreeFlann(pcd_ideal)
    for i in surviving_indices:
        [k, idx, _] = kdtree.search_knn_vector_3d(points[i], K_NEIGHBORS)
        avg_noise = np.mean(raw_noise[idx], axis=0)
        smoothed_noise[i] = avg_noise * np.sqrt(K_NEIGHBORS)

surviving_points = points[surviving_indices]
surviving_noise = smoothed_noise[surviving_indices]
noisy_points = surviving_points + surviving_noise

pcd_simulated_noise = o3d.geometry.PointCloud()
pcd_simulated_noise.points = o3d.utility.Vector3dVector(noisy_points)
print(f"Simulation successful! {len(surviving_points)} wavy points generated.")


Calculating AoI...
Dropping points based on Gaussian Model...
Dropout Complete: 7329 points deleted.
Generating Raw TV-Static Gaussian Noise...
Applying Spatial Moving Average (Simulating Firmware Blur for Wavy Effect)...
Simulation successful! 8028 wavy points generated.


In [154]:
# ==========================================
# 5. VISUALIZATION
# ==========================================
pcd_ideal.paint_uniform_color([1, 0, 0])             # Red: Ideal CAD
pcd_simulated_noise.paint_uniform_color([0, 1, 0])   # Green: Mathematical Noise

# Translate them apart horizontally by 50mm
pcd_ideal_viz = copy.deepcopy(pcd_ideal).translate([-50, 0, 0])
pcd_noise_viz = copy.deepcopy(pcd_simulated_noise).translate([50, 0, 0])

viz_list = [pcd_ideal_viz, pcd_noise_viz]
print("Red (Left): Ideal CAD | Green (Right): Simulated (Dropout + Wavy Noise)")
print("Close Open3D window to continue.")

o3d.visualization.draw_geometries(viz_list, window_name=f"Physics Simulation - View {TARGET_VIEWPOINT}")


Red (Left): Ideal CAD | Green (Right): Simulated (Dropout + Wavy Noise)
Close Open3D window to continue.


In [155]:
# ==========================================
# 5.5 3-WAY VISUALIZATION (INCLUDING ACTUAL SCAN)
# ==========================================
# Set up the paths to your real processed sensor data here:
EXPERIMENT = "test_5_090angle"  # Update this if your real data is in a different folder!
PROCESSED_DIR = f"processed_data/{EXPERIMENT}"
FULL_TRANSFORM_SAVE_DIR = f'evaluation_result/{EXPERIMENT}/merge_full_transformation.npy'

real_path = os.path.join(PROCESSED_DIR, f"viewpoint_simulated_{TARGET_VIEWPOINT}.pcd")

if os.path.exists(real_path) and os.path.exists(FULL_TRANSFORM_SAVE_DIR):
    pcd_real = o3d.io.read_point_cloud(real_path)
    merge_full_transformation = np.load(FULL_TRANSFORM_SAVE_DIR)
    
    # Move real scan from Base -> CAD Space -> Camera Space
    T_target_to_object = np.linalg.inv(merge_full_transformation)
    pcd_real.transform(T_target_to_object)
    pcd_real.transform(T_obj_to_cam)
    
    pcd_ideal.paint_uniform_color([1, 0, 0])         # Red: Ideal CAD
    pcd_simulated_noise.paint_uniform_color([0, 1, 0]) # Green: Mathematical Noise
    pcd_real.paint_uniform_color([0, 0.2, 0.8])      # Blue: Real Scan
    
    # Translate them apart horizontally by 100mm
    pcd_ideal_viz = copy.deepcopy(pcd_ideal).translate([-100, 0, 0])
    pcd_noise_viz = copy.deepcopy(pcd_simulated_noise).translate([0, 0, 0])
    pcd_real_viz = copy.deepcopy(pcd_real).translate([100, 0, 0])
    pcd_noise_viz.estimate_normals()
    pcd_real_viz.estimate_normals()
    
    viz_list = [pcd_ideal_viz, pcd_noise_viz, pcd_real_viz]
    print("Red (Left): Ideal CAD | Green (Center): Simulated | Blue (Right): Real Sensor")
    print("Close Open3D window to continue.")
    
    o3d.visualization.draw_geometries(viz_list, window_name=f"3-Way Physics Comparison - View {TARGET_VIEWPOINT}")
else:
    print(f"Could not find real scan data at: {real_path}")
    print(f"Or missing transformation matrix at: {FULL_TRANSFORM_SAVE_DIR}")


Red (Left): Ideal CAD | Green (Center): Simulated | Blue (Right): Real Sensor
Close Open3D window to continue.


In [148]:
# ==========================================
# 6. BATCH PROCESS ALL VIEWPOINTS
# ==========================================
import os
import glob
import math
import numpy as np
import open3d as o3d

# CONFIGURATION
WORKPIECES = ["TH0011AV", "TH0012AV", "TH0021AV", "TH0022AV", "TH0031AV", "TH0032AV"]

SIGMA_AOI = 30.0  
BASE_SIGMA_XY = 1.0  
BASE_SIGMA_Z = 2.0   
K_NEIGHBORS = 300

def calculate_dropout_probability(aoi_degrees):
    survival = np.exp(-(aoi_degrees**2) / (2 * (SIGMA_AOI**2)))
    return 1.0 - survival

for workpiece in WORKPIECES:
    print(f"\n==========================================")
    print(f"Processing Workpiece: {workpiece}")
    print(f"==========================================")
    
    SIM_DIR = f"viewpoints_candidate/testing_data/{workpiece}"
    OUT_DIR = f"simulation/{workpiece}"
    
    if not os.path.exists(OUT_DIR):
        os.makedirs(OUT_DIR)
        
    sim_files = glob.glob(os.path.join(SIM_DIR, "viewpoint_simulated_*.pcd"))
    print(f"Found {len(sim_files)} viewpoints for {workpiece}.")
    
    for sim_path in sim_files:
        basename = os.path.basename(sim_path)
        view_idx_str = basename.replace("viewpoint_simulated_", "").replace(".pcd", "")
        view_idx = int(view_idx_str)
        
        pose_path = os.path.join(SIM_DIR, f"viewpoint_pose_{view_idx}.npy")
        if not os.path.exists(pose_path):
            continue
            
        # 1. LOAD & TRANSFORM TO CAMERA SPACE
        pcd_ideal = o3d.io.read_point_cloud(sim_path)
        
        if not pcd_ideal.has_normals():
            pcd_ideal.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=10.0, max_nn=30))
            pcd_ideal.orient_normals_towards_camera_location(camera_location=np.array([0., 0., 0.]))
            normals = np.asarray(pcd_ideal.normals)
            pcd_ideal.normals = o3d.utility.Vector3dVector(-normals)
            
        T_cam_to_obj = np.load(pose_path)
        T_obj_to_cam = np.linalg.inv(T_cam_to_obj)
        pcd_ideal.transform(T_obj_to_cam)
        
        points = np.asarray(pcd_ideal.points)
        normals = np.asarray(pcd_ideal.normals)
        
        # 2. CALCULATE AOI
        camera_dir = -points
        camera_dir_norms = np.linalg.norm(camera_dir, axis=1, keepdims=True)
        camera_dir_norm = camera_dir / camera_dir_norms
        normal_norms = np.linalg.norm(normals, axis=1, keepdims=True)
        normal_norm_vecs = normals / normal_norms
        
        cos_thetas = np.clip(np.sum(camera_dir_norm * normal_norm_vecs, axis=1), -1.0, 1.0)
        aois = np.degrees(np.arccos(cos_thetas))
        aois = np.where(aois > 90, 180 - aois, aois)
        
        # 3. MANUAL GAUSSIAN DROPOUT
        idx_by_bin = {i: [] for i in range(91)}
        for i in range(len(points)):
            bin_idx = int(np.round(aois[i]))
            if bin_idx > 90: bin_idx = 90
            idx_by_bin[bin_idx].append(i)
            
        surviving_indices = []
        for bin_idx, p_idx_list in idx_by_bin.items():
            if len(p_idx_list) == 0:
                continue
            prob = calculate_dropout_probability(bin_idx)
            num_to_drop = int(len(p_idx_list) * prob)
            np.random.shuffle(p_idx_list)
            surviving = p_idx_list[num_to_drop:]
            surviving_indices.extend(surviving)
            
        # 4. SPATIAL WAVY NOISE
        raw_noise_x = np.random.normal(0, BASE_SIGMA_XY, size=len(points))
        raw_noise_y = np.random.normal(0, BASE_SIGMA_XY, size=len(points))
        raw_noise_z = np.random.normal(0, BASE_SIGMA_Z, size=len(points))
        raw_noise = np.column_stack((raw_noise_x, raw_noise_y, raw_noise_z))
        
        kdtree = o3d.geometry.KDTreeFlann(pcd_ideal)
        smoothed_noise = np.zeros_like(raw_noise)
        
        for i in surviving_indices:
            [k, idx, _] = kdtree.search_knn_vector_3d(points[i], K_NEIGHBORS)
            avg_noise = np.mean(raw_noise[idx], axis=0)
            smoothed_noise[i] = avg_noise * np.sqrt(K_NEIGHBORS)
            
        surviving_points = points[surviving_indices]
        surviving_noise = smoothed_noise[surviving_indices]
        noisy_points = surviving_points + surviving_noise
        
        # 5. TRANSFORM BACK & SAVE
        pcd_simulated_noise = o3d.geometry.PointCloud()
        pcd_simulated_noise.points = o3d.utility.Vector3dVector(noisy_points)
        
        pcd_simulated_noise.transform(T_cam_to_obj)
        
        out_path = os.path.join(OUT_DIR, f"viewpoint_simulated_noise_{view_idx}.pcd")
        o3d.io.write_point_cloud(out_path, pcd_simulated_noise)

print(f"\nALL BATCHES DONE!")



Processing Workpiece: TH0011AV
Found 216 viewpoints for TH0011AV.

Processing Workpiece: TH0012AV
Found 216 viewpoints for TH0012AV.

Processing Workpiece: TH0021AV
Found 216 viewpoints for TH0021AV.

Processing Workpiece: TH0022AV
Found 216 viewpoints for TH0022AV.

Processing Workpiece: TH0031AV
Found 216 viewpoints for TH0031AV.

Processing Workpiece: TH0032AV
Found 216 viewpoints for TH0032AV.

ALL BATCHES DONE!
